In [66]:
from dotenv import load_dotenv
load_dotenv()

True

In [67]:
from google import genai
gemini_client = genai.Client() # picks up the API key from the env variable GEMINI_API_KEY

In [68]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [69]:
assistant = RAGBase(
    index = index,
    llm_client=gemini_client
)

In [70]:
assistant.rag("How do I run Ollama locally?")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 20.093378257s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'},

In [ ]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
from google.genai import types

# Define the schema explicitly, just like in the lesson
search_declaration = types.FunctionDeclaration(
    name="search",
    description="Search the FAQ database for entries matching the given query.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
    }
)

# You must wrap the FunctionDeclaration inside a Tool object
search_tool = types.Tool(
    function_declarations=[search_declaration]
)

In [ ]:
messages = [
    {'role': 'user', 'parts': [{'text': 'I just dicovered the course. Can I join it?'}]},
]

In [ ]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [ ]:
from google.genai import types

response = gemini_client.models.generate_content(
    model="gemini-3.6-flash",
    contents=messages,
    config=types.GenerateContentConfig(
        tools=[search_tool]
    )
)

In [ ]:
call = response.function_calls[0]
call.name

'search'

In [ ]:
results = search(**call.args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub

In [ ]:
messages.append(response.candidates[0].content)

messages.append(
    types.Content(
        role="user",
        parts=[
            types.Part.from_function_response(
                name=call.name,
                response={"result": results}, # Must be passed as a dictionary
            )
        ]
    )
)

In [ ]:
response = gemini_client.models.generate_content(
    model="gemini-3.6-flash",
    contents=messages,
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCTIONS,
        tools=[search_tool]
    )
)

print(response.text)

Yes, you can still join!

If you want to receive a certificate, you will need to submit your project while submissions are still being accepted. All course videos and GitHub materials are available, so you can start learning and working through the materials whenever you want.


In [ ]:
usage = response.usage_metadata

print("Input Tokens:", usage.prompt_token_count)
print("Output Tokens:", usage.candidates_token_count)

def calculate_gemini_flash_price(input_tokens, output_tokens):
    # Standard pricing rates for gemini-3.6-flash (per million tokens)
    INPUT_PRICE_PER_MILLION = 0.75
    OUTPUT_PRICE_PER_MILLION = 4.50

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

cost_info = calculate_gemini_flash_price(usage.prompt_token_count, usage.candidates_token_count)
print("Total cost: $", round(cost_info["total_cost"], 8))

Input Tokens: 987
Output Tokens: 53
Total cost: $ 0.00097875


*
*
*
Conversation History

1. Making a request (query) to the LLM <-- first request
2. LLM decides to invoke Search('with parameters')
3. Getting results as Search() output
4. Sending the results back to the LLM <-- a second request
5. LLM processes the results
6. LLM gives an answer